In [37]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [38]:
RAW_DATA = "../data/raw_data.csv"

In [39]:
df = pd.read_csv(RAW_DATA)
df.head()

,id,bucket,media_type,title,native_title,description,country_of_origin,format,status,year,average_score,popularity,genres,tags,synonyms,relations,image_url,is_adult
0,1,anime,ANIME,Cowboy Bebop,カウボーイビバップ,"Enter a world in the distant future, where Bou...",JP,TV,FINISHED,1998.0,86,451502,"Action,Adventure,Drama,Sci-Fi","Space,Crime,Episodic,Ensemble Cast,Primarily A...","카우보이 비밥,קאובוי ביבופ,คาวบอย บีบ๊อป,Ковбой Бибо...","[{""relationType"": ""SIDE_STORY"", ""node"": {""id"":...",https://s4.anilist.co/file/anilistcdn/media/an...,False
1,5,anime,ANIME,Cowboy Bebop: The Movie - Knockin' on Heaven's...,カウボーイビバップ天国の扉,"As the Cowboy Bebop crew travels the stars, th...",JP,MOVIE,FINISHED,2001.0,82,82162,"Action,Drama,Mystery,Sci-Fi","Terrorism,Primarily Adult Cast,Philosophy,Crim...","Cowboy Bebop Movie,Cowboy Bebop: The Movie,Cow...","[{""relationType"": ""PARENT"", ""node"": {""id"": 1, ...",https://s4.anilist.co/file/anilistcdn/media/an...,False
2,6,anime,ANIME,Trigun,TRIGUN,Vash the Stampede is a wanted man with a habit...,JP,TV,FINISHED,1998.0,80,164676,"Action,Adventure,Comedy,Drama,Sci-Fi","Guns,Fugitive,Male Protagonist,Philosophy,Prim...","トライガン,ไทรกัน,Триган","[{""relationType"": ""ADAPTATION"", ""node"": {""id"":...",https://s4.anilist.co/file/anilistcdn/media/an...,False
3,7,anime,ANIME,Witch Hunter ROBIN,Witch Hunter ROBIN,Robin Sena is a powerful craft user drafted in...,JP,TV,FINISHED,2002.0,68,23515,"Action,Drama,Mystery,Supernatural","Conspiracy,Police,Female Protagonist,Magic,Urb...","ウイッチハンターロビン,WHR",[],https://s4.anilist.co/file/anilistcdn/media/an...,False
4,8,anime,ANIME,Beet the Vandel Buster,冒険王ビィト,It is the dark century and the people are suff...,JP,TV,FINISHED,2004.0,65,3119,"Adventure,Fantasy,Supernatural","Shounen,Spearplay,Swordplay",Adventure King Beet,"[{""relationType"": ""SEQUEL"", ""node"": {""id"": 112...",https://s4.anilist.co/file/anilistcdn/media/an...,False


In [40]:
print(df.shape)
df.info()

(177156, 18)
<class 'pandas.DataFrame'>
RangeIndex: 177156 entries, 0 to 177155
Data columns (total 18 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 177156 non-null  int64  
 1   bucket             177156 non-null  str    
 2   media_type         177156 non-null  str    
 3   title              177156 non-null  str    
 4   native_title       176655 non-null  str    
 5   description        139713 non-null  str    
 6   country_of_origin  177156 non-null  str    
 7   format             177108 non-null  str    
 8   status             177102 non-null  str    
 9   year               177147 non-null  float64
 10  average_score      177156 non-null  int64  
 11  popularity         177156 non-null  int64  
 12  genres             171193 non-null  str    
 13  tags               143044 non-null  str    
 14  synonyms           98426 non-null   str    
 15  relations          177156 non-null  str    
 16  

In [41]:
df.duplicated().any()
#There are duplicate entries possibly due to the script crashing in between and me running the script again

np.True_

In [42]:
df.duplicated().sum()

np.int64(30537)

In [43]:
# Removing duplicated data 
df = df.drop_duplicates(subset=['id'], keep='last')
print(df.shape)

(146350, 18)


In [44]:
# manga_mask = df['bucket'].isin(['manga', 'manhwa', 'manhua'])
print(df['year'].value_counts().sort_index())

year
1917.0        1
1924.0        1
1925.0        1
1926.0        1
1927.0        1
1928.0        3
1929.0        4
1930.0        3
1931.0       10
1932.0        4
1933.0        8
1934.0        6
1935.0        9
1936.0        6
1937.0        1
1938.0        1
1939.0        1
1940.0        1
1941.0        1
1942.0        2
1943.0        2
1945.0        1
1946.0        1
1947.0        1
1948.0        1
1949.0        1
1950.0        6
1951.0       10
1952.0        8
1953.0        7
1954.0       21
1955.0       17
1956.0       32
1957.0       29
1958.0       56
1959.0       43
1960.0       40
1961.0       43
1962.0       40
1963.0       49
1964.0       73
1965.0       85
1966.0       84
1967.0      116
1968.0      129
1969.0      145
1970.0      155
1971.0      132
1972.0      148
1973.0      156
1974.0      160
1975.0      200
1976.0      240
1977.0      307
1978.0      255
1979.0      277
1980.0      260
1981.0      301
1982.0      318
1983.0      331
1984.0      294
1985.0      403
198

checked for suspicious dips in manga/anime by year to see if the fault is in my script crashing but the distribution seems fair there is a decline post 2021 is due to anilist being a community driven service so it takes time to add newer titles and 2026 especially is incomplete 

In [45]:
# Bucket distribution to see what was actually collected
print(df['bucket'].value_counts())

# Titles with no description - these will be weak recommendations so should remove them ideally but I will try filling the data with some template using the other fields
no_desc = df['description'].isna().sum()
print(f"\nTitles missing description: {no_desc} ({no_desc/len(df)*100:.1f}%)")
print("These will be filtered out before embedding")

bucket
manga     107875
anime      19960
manhwa     13622
manhua      4893
Name: count, dtype: int64

Titles missing description: 32340 (22.1%)
These will be filtered out before embedding


In [46]:
df['status' ].value_counts().sort_values(ascending=False)

status
FINISHED            126029
RELEASING            19553
CANCELLED              558
NOT_YET_RELEASED       158
Name: count, dtype: int64

In [47]:
def synthesize_description(row) -> str:
    """
    Builds a minimal proxy description from structured fields
    when the real description is missing.
    """
    def safe_get(field):
        val = row[field]
        return "" if pd.isna(val) else str(val).strip()

    status = safe_get("status")
    bucket = safe_get("bucket")
    genres_raw = safe_get("genres")
    tags_raw = safe_get("tags")

    parts = []

    status_map = {
        "FINISHED": "finished",
        "RELEASING": "ongoing",
        "CANCELLED": "cancelled",
        "NOT_YET_RELEASED": "upcoming",
    }
    status_word = status_map.get(status, "")

    if status_word and bucket:
        parts.append(f"A {status_word} {bucket}")
    elif bucket:
        parts.append(f"A {bucket}")
    elif status_word:
        parts.append(f"A {status_word} title")
    else:
        parts.append("A title")

    genres = [g.strip() for g in genres_raw.split(",") if g.strip()]
    tags = [t.strip() for t in tags_raw.split(",") if t.strip()][:5]

    if genres and tags:
        parts.append(f"with genres including {', '.join(genres)}")
        parts.append(f"and themes such as {', '.join(tags)}")
    elif genres:
        parts.append(f"with genres including {', '.join(genres)}")
    elif tags:
        parts.append(f"with themes such as {', '.join(tags)}")
    else:
        parts.append("with no additional metadata available")

    return " ".join(parts) + "."

In [48]:
# check to see if the transformation was applied here is an example of title with no description
empty_row = df[df['description'].isna()].iloc[10]
print(empty_row)

id                                                              186856
bucket                                                           anime
media_type                                                       ANIME
title                  Hito no Kurashi no Hyakumannen: Mani Mani March
native_title                                       人のくらしの百万年 マニ・マニ・マーチ
description                                                        NaN
country_of_origin                                                   JP
format                                                           MOVIE
status                                                        FINISHED
year                                                            1968.0
average_score                                                        0
popularity                                                          22
genres                                                             NaN
tags                                                               NaN
synony

In [49]:
# Create mask BEFORE filling - captures which rows had missing descriptions
mask = df['description'].isna() | (df['description'].str.strip() == "")

# Tag synthetic rows as True/False boolean
df['description_is_synthetic'] = mask

# Now apply synthetic descriptions using the same mask
df.loc[mask, 'description'] = df[mask].apply(
    synthesize_description, axis=1
)

# Verify no nulls remain
print(f"Remaining missing descriptions: {df['description'].isna().sum()}")

# Quick sanity check - look at a few synthetic ones
print("\nSample synthetic descriptions:")
print(df[mask]['description'].head(5).to_string())

Remaining missing descriptions: 0

Sample synthetic descriptions:
5098      A finished anime with genres including Fantasy.
5101       A finished anime with genres including Action.
5117    A finished anime with no additional metadata a...
5129    A finished anime with no additional metadata a...
5151    A finished anime with genres including Slice o...


In [55]:
# checking if the row is empty or if its filled with description now
df[df['id']==186856]['description']

5339    A finished anime with no additional metadata a...
Name: description, dtype: str

In [51]:
print(f"Total titles: {df.shape[0]}")
print(f"Synthetic descriptions: {df['description_is_synthetic'].sum()}")
print(f"Real descriptions: {(~df['description_is_synthetic']).sum()}")
print(f"Remaining nulls in description: {df['description'].isna().sum()}")

Total titles: 146350
Synthetic descriptions: 32340
Real descriptions: 114010
Remaining nulls in description: 0


In [ ]:
# Save the clean dataset
df.to_csv("../data/raw_data_clean.csv", index=False)
print(f"Clean dataset saved: {df.shape[0]} unique titles")